<a href="https://colab.research.google.com/github/madelsu/Dynamic_Hyperparameter_Optimization_LLM/blob/main/(e)Dynamic_Bayesian_Optimization/Dynamic(look_up)_Bayesian_Op.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymysql
import pandas as pd
from sqlalchemy import create_engine, text  # 👈 make sure text is imported here

# Connection details
host = ""
port =
user = "root"
password = ""
database = "db_icsr_assessment_manuela"

# Create SQLAlchemy engine
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# Test the connection by listing tables
with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES;"))
    tables = [row[0] for row in result]

print("✅ Connected successfully!")
print("Available tables:", tables)

# === Read the entire table 'icsr_assessment_import' ===
query = text("SELECT * FROM icsr_assessment_import;")

with engine.connect() as conn:
    hum = pd.read_sql(query, conn)

# === Display results ===
print("✅ Data successfully loaded from 'icsr_assessment_import'")
print("Number of rows:", len(hum))
print("Columns:", list(hum.columns))
hum.head()  # preview first few rows

In [ ]:
# === Read the entire table 'hp_results2' ===
query = text("SELECT * FROM hp_results2;")

with engine.connect() as conn:
    llm = pd.read_sql(query, conn)

# === Display results ===
print("✅ Data successfully loaded from 'hp_results2'")
print("Number of rows:", len(llm))
print("Columns:", list(llm.columns))
llm.head()  # preview first few rows

In [ ]:
# --- 1️⃣ Import required libraries ---
import pandas as pd
from IPython.display import display
from google.colab import files

# --- 2️⃣ Upload your file ---
print("📤 Please select your CSV file to upload")
uploaded = files.upload()  # opens a file picker in Colab

# --- 3️⃣ Read the uploaded CSV into pandas ---
# Automatically use the uploaded filename
filename = list(uploaded.keys())[0]
agreement_df= pd.read_csv(filename, encoding="latin1")

# --- 4️⃣ Set display options for better visualization ---
pd.set_option('display.max_rows', None)    # show all rows
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.width', 200)        # wider view
pd.set_option('display.colheader_justify', 'center')

# --- 5️⃣ Display the DataFrame ---
display(agreement_df)

In [ ]:
print("Number of unique cases:", agreement_df["case_id"].nunique())

In [ ]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Optional

def create_time_samples_per_case(agreement_df: pd.DataFrame,
                                 target_min: int = 50,
                                 target_max: int = 60,
                                 n_samples: int = 10,
                                 group_key: Tuple[str, str, str] = ("drug", "event", "temperature"),
                                 random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Greedy packing of *whole cases* into ~equal-sized samples.
    Each sample will have between target_min and target_max rows (if possible).
    Sampling is done per case_id (a case stays intact in one sample).

    Returns:
      - sample_assignments: agreement_df + sample_id column
      - samples_meta: per-sample summary (n_rows, n_cases, mean_agreement, etc.)
    """
    rng = np.random.default_rng(random_state)

    # Validate required columns
    required_cols = ["case_id", "question_agreement_rate"] + list(group_key)
    missing = [c for c in required_cols if c not in agreement_df.columns]
    if missing:
        raise ValueError(f"agreement_df is missing required columns: {missing}")

    # Compute per-case stats
    case_groups = (agreement_df
                   .groupby("case_id", dropna=False)
                   .agg(n_rows=("case_id", "size"),
                        mean_agreement=("question_agreement_rate", "mean"))
                   .reset_index())

    # Shuffle cases
    case_groups = case_groups.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    # Prepare outputs
    sample_bins: Dict[int, List[str]] = {i: [] for i in range(1, n_samples + 1)}  # case_id lists
    sample_sizes = {i: 0 for i in range(1, n_samples + 1)}

    # Greedy fill by case
    for _, row in case_groups.iterrows():
        cid = row["case_id"]
        csize = int(row["n_rows"])
        smallest = sorted(sample_sizes.items(), key=lambda kv: kv[1])[0][0]
        sample_bins[smallest].append(cid)
        sample_sizes[smallest] += csize

    # Map case_id -> sample_id
    case_to_sample = {cid: sid for sid, clist in sample_bins.items() for cid in clist}

    # Assign back to full dataframe
    agreement_df = agreement_df.copy()
    agreement_df["sample_id"] = agreement_df["case_id"].map(case_to_sample)

    # Build per-sample meta
    meta_rows = []
    for sid in range(1, n_samples + 1):
        sample_df = agreement_df.loc[agreement_df["sample_id"] == sid]
        meta_rows.append({
            "sample_id": sid,
            "n_rows": len(sample_df),
            "n_cases": sample_df["case_id"].nunique(),
            "mean_agreement": float(sample_df["question_agreement_rate"].mean()) if len(sample_df) else np.nan,
            "median_agreement": float(sample_df["question_agreement_rate"].median()) if len(sample_df) else np.nan,
        })
    samples_meta = pd.DataFrame(meta_rows).sort_values("sample_id").reset_index(drop=True)

    return agreement_df, samples_meta

In [ ]:
# 1) Make samples (e.g., ~55 rows per time point, 10 time points)
sampled_df, samples_meta = create_time_samples_per_case(
    agreement_df,
    target_min=50,
    target_max=60,
    n_samples=10,
    group_key=("case_id", "temperature"),
    random_state=123
)

# 2) Inspect balance and means
display(samples_meta)
# Columns: sample_id | n_rows | n_groups | mean_agreement | median_agreement

# 3) sampled_df now has a 'sample_id' column to simulate time (Day = sample_id)
#    Keep this for the next step.

In [ ]:
!pip install scikit-optimize

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List
from skopt import gp_minimize
from skopt.space import Categorical
from skopt.utils import use_named_args

# -----------------------------------------------------------
# 1) Choose which hyperparameter columns to let BO consider
#    (only those that actually exist in agreement_df / sampled_df)
#    From your columns, these are present:
#       temperature, frequency_penalty, max_new_tokens
# -----------------------------------------------------------
CANDIDATE_PARAM_COLS: List[str] = [
    "temperature",
    "frequency_penalty",
    "max_new_tokens",
    # add more only if they EXIST in your sampled_df (e.g., "top_p")
]

# -----------------------------------------------------------
# 2) Evaluation: exact lookup of mean %agreement for a param combo
# -----------------------------------------------------------
def _mask_params(df: pd.DataFrame, params: dict) -> pd.Series:
    m = pd.Series(True, index=df.index)
    for k, v in params.items():
        if k in df.columns:
            m &= (df[k] == v)
    return m

def evaluate_lookup(sample_df: pd.DataFrame, params: dict) -> float:
    """
    Returns mean question_agreement_rate for rows in sample_df
    that match the chosen (categorical) params exactly.
    If no match, returns 0.0 (discourages BO from picking it again).
    """
    mask = _mask_params(sample_df, params)
    matched = sample_df.loc[mask]
    if matched.empty:
        return 0.0
    return float(matched["question_agreement_rate"].mean())

# -----------------------------------------------------------
# 3) Build per-sample categorical search space
#    (drop params that have < 2 unique values in that sample)
# -----------------------------------------------------------
def build_search_space_for_sample(sample_df: pd.DataFrame,
                                  param_cols: List[str]) -> List[Categorical]:
    dims = []
    for col in param_cols:
        if col in sample_df.columns:
            vals = sorted(sample_df[col].dropna().unique().tolist())
            # keep only if there is actual variation
            if len(vals) >= 2:
                dims.append(Categorical(vals, name=col))
    return dims

# -----------------------------------------------------------
# 4) Run BO over all samples (days) in lookup mode
# -----------------------------------------------------------
def run_lookup_bo_over_time(sampled_df: pd.DataFrame,
                            samples_meta: pd.DataFrame,
                            param_cols: List[str] = CANDIDATE_PARAM_COLS,
                            n_calls: int = 25,
                            n_initial_points: int = 8,
                            random_state: int = 2025):
    bo_summ_rows = []
    bo_trials_rows = []

    for sid in sorted(samples_meta["sample_id"].tolist()):
        current_sample_df = sampled_df.loc[sampled_df["sample_id"] == sid].copy()
        # Build categorical space based on what actually exists in THIS sample
        space = build_search_space_for_sample(current_sample_df, param_cols)

        # If nothing varies, just record the mean and move on
        if len(space) == 0:
            mean_ag = float(current_sample_df["question_agreement_rate"].mean()) if len(current_sample_df) else np.nan
            bo_summ_rows.append({
                "sample_id": sid,
                "best_mean_agreement": mean_ag,
                # no params to report
            })
            continue

        @use_named_args(space)
        def objective(**params):
            mean_score = evaluate_lookup(current_sample_df, params)
            bo_trials_rows.append({"sample_id": sid, **params, "mean_agreement": mean_score})
            return -mean_score  # minimize

        res = gp_minimize(
            func=objective,
            dimensions=space,
            n_calls=n_calls,
            n_initial_points=n_initial_points,
            random_state=random_state + sid
        )

        best_score = -res.fun
        best_params = {dim.name: val for dim, val in zip(space, res.x)}
        bo_summ_rows.append({
            "sample_id": sid,
            "best_mean_agreement": float(best_score),
            **best_params
        })

    bo_summary = pd.DataFrame(bo_summ_rows).sort_values("sample_id").reset_index(drop=True)
    bo_trials = pd.DataFrame(bo_trials_rows)
    return bo_summary, bo_trials

# -----------------------------------------------------------
# 5) Run it
# -----------------------------------------------------------
bo_summary, bo_trials = run_lookup_bo_over_time(
    sampled_df=sampled_df,
    samples_meta=samples_meta,
    param_cols=CANDIDATE_PARAM_COLS,
    n_calls=25,
    n_initial_points=8,
    random_state=2025
)

display(bo_summary.head())
display(bo_trials.head())

# -----------------------------------------------------------
# 6) Plot performance-over-time + (optional) optimal params
# -----------------------------------------------------------
x = bo_summary["sample_id"].values
y = bo_summary["best_mean_agreement"].values

plt.figure()
plt.plot(x, y, marker="o")
plt.xlabel("Time (sample ID ~ day)")
plt.ylabel("Best mean % agreement")
plt.title("Performance over time — Lookup-only BO")
plt.grid(True)
plt.show()

# Plot any optimal params that exist
for col in CANDIDATE_PARAM_COLS:
    if col in bo_summary.columns:
        plt.figure()
        plt.plot(x, bo_summary[col].values, marker="o")
        plt.xlabel("Time (sample ID ~ day)")
        plt.ylabel(f"Optimal {col}")
        plt.title(f"Optimal {col} over time (lookup-only)")
        plt.grid(True)
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# --- Prepare data ---
df_plot = bo_summary.copy()
df_plot["temperature"] = pd.to_numeric(df_plot["temperature"], errors="coerce")
df_plot["best_mean_agreement"] = pd.to_numeric(df_plot["best_mean_agreement"], errors="coerce")

# --- Better visual style ---
sns.set(style="whitegrid", font_scale=1.2)

plt.figure(figsize=(10, 6))

# --- Scatterplot with connecting line ---
points = plt.scatter(
    df_plot["sample_id"],
    df_plot["best_mean_agreement"],
    c=df_plot["temperature"],
    cmap="coolwarm",
    s=120,
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9
)
plt.plot(
    df_plot["sample_id"],
    df_plot["best_mean_agreement"],
    color="gray",
    linestyle="--",
    alpha=0.5
)

# --- Colorbar and labels ---
cbar = plt.colorbar(points)
cbar.set_label("BO Optimal Temperature", fontsize=12)

plt.xlabel("Time (sample_id)", fontsize=12)
plt.ylabel("Best Mean % Agreement", fontsize=12)
plt.title("Best Mean Agreement Over Time Colored by Temperature", fontsize=14, weight='bold')

# --- Adjust axes for better balance ---
plt.ylim(0.95, 1.1)  # since all points are near 1, zoom in to show variation
plt.xticks(df_plot["sample_id"])
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# 7) Run multiple samplings to show stability
# -----------------------------------------------------------
n_repeats = 10
all_runs = []

for repeat_idx in range(n_repeats):
    bo_summary, _ = run_lookup_bo_over_time(
        sampled_df=sampled_df,
        samples_meta=samples_meta,
        param_cols=CANDIDATE_PARAM_COLS,
        n_calls=25,
        n_initial_points=8,
        random_state=2025 + repeat_idx  # different seed for each repeat
    )
    all_runs.append(bo_summary["best_mean_agreement"].values)

all_runs = np.array(all_runs)  # shape: (n_repeats, n_samples)
x = bo_summary["sample_id"].values

# -----------------------------------------------------------
# 8) Combined plot
# -----------------------------------------------------------
plt.figure(figsize=(10, 6))

# Plot each run with some transparency
for run in all_runs:
    plt.plot(x, run, color="skyblue", alpha=0.5)

# Plot mean across all repeats
plt.plot(x, all_runs.mean(axis=0), color="blue", marker="o", label="Mean over 10 repeats")

plt.xlabel("Time (sample ID ~ day)")
plt.ylabel("Best mean % agreement")
plt.title("Lookup-only BO: Multiple samplings (10x) — shows minimal sampling impact")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 6))

# --- Plot each run with transparency ---
for run in all_runs:
    plt.plot(x, run, color="skyblue", alpha=0.3)  # faint lines

# --- Mean line ---
mean_run = all_runs.mean(axis=0)
plt.plot(x, mean_run, color="black", linestyle="--", alpha=0.7, label="Mean over 10 repeats")

# --- Scatter with temperature as color ---
temp_values = bo_summary["temperature"].values
sc = plt.scatter(x, mean_run, c=temp_values, cmap="coolwarm", s=100, edgecolor="black", linewidth=0.5, zorder=5)

# --- Annotate each dot with temperature ---
for xi, yi, temp in zip(x, mean_run, temp_values):
    plt.text(xi, yi + 0.001, f"{temp:.1f}", ha='center', va='bottom', fontsize=9, rotation=0)

# --- Colorbar for temperature ---
cbar = plt.colorbar(sc)
cbar.set_label("BO Optimal Temperature")

# --- Labels, limits, grid ---
plt.xlabel("Time (sample ID ~ day)")
plt.ylabel("Best mean % agreement")
plt.ylim(0.95, 1.1)
plt.title("Lookup-only BO: Multiple samplings (10x) with Temperature on Points")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Prepare data ---
df_temp = bo_summary.copy()
df_temp["temperature"] = pd.to_numeric(df_temp["temperature"], errors="coerce")
x = df_temp["sample_id"].values
y = df_temp["temperature"].values

# --- Plot style ---
sns.set(style="whitegrid", font_scale=1.2)
plt.figure(figsize=(12, 6))

# --- Line plot for temperature trend ---
plt.plot(x, y, color="gray", linestyle="--", alpha=0.6, label="Temperature Trend")

# --- Scatter points colored by temperature ---
sc = plt.scatter(x, y, c=y, cmap="coolwarm", s=120, edgecolor="black", linewidth=0.6)

# --- Colorbar ---
cbar = plt.colorbar(sc)
cbar.set_label("Temperature")

# --- Labels and title ---
plt.xlabel("Time (sample_id)")
plt.ylabel("Temperature")
plt.title("Variation of Temperature Over Time")
plt.grid(True, linestyle="--", alpha=0.4)

# --- Optional: annotate each point with value (1 decimal) ---
for xi, yi in zip(x, y):
    plt.text(xi, yi + 0.02, f"{yi:.1f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# --- Prepare data ---
x = bo_summary["sample_id"].values
mean_agreement = all_runs.mean(axis=0)
y = mean_agreement
n = len(y)

# Initialize DataFrame
df_flags = pd.DataFrame({
    "sample_id": x,
    "mean_agreement": y,
    "Z_lower": np.nan,
    "Z_upper": np.nan,
    "IQR_lower": np.nan,
    "IQR_upper": np.nan,
    "Rolling_lower": np.nan,
    "Rolling_upper": np.nan,
    "Z_flag": "",
    "IQR_flag": "",
    "Rolling_flag": ""
})

# Parameters
rolling_window = 3
z_thresh = 2
iqr_mult = 1.5
rolling_mult = 2
epsilon = 1e-8

# --- Compute cumulative thresholds and flags ---
for i in range(n):
    y_cum = y[:i+1]

    # Z-score
    mean_val = np.mean(y_cum)
    std_val = np.std(y_cum) + epsilon
    z_lower = mean_val - z_thresh * std_val
    z_upper = mean_val + z_thresh * std_val
    df_flags.loc[i, "Z_lower"] = z_lower
    df_flags.loc[i, "Z_upper"] = z_upper
    df_flags.loc[i, "Z_flag"] = "Yes" if y[i] < z_lower or y[i] > z_upper else ""

    # IQR
    Q1, Q3 = np.percentile(y_cum, [25, 75])
    IQR = Q3 - Q1
    iqr_lower = Q1 - iqr_mult * IQR
    iqr_upper = Q3 + iqr_mult * IQR
    df_flags.loc[i, "IQR_lower"] = iqr_lower
    df_flags.loc[i, "IQR_upper"] = iqr_upper
    df_flags.loc[i, "IQR_flag"] = "Yes" if y[i] < iqr_lower or y[i] > iqr_upper else ""

    # Rolling
    if i < rolling_window-1:
        roll_vals = y_cum
    else:
        roll_vals = y_cum[-rolling_window:]
    roll_mean = np.mean(roll_vals)
    roll_std = np.std(roll_vals) + epsilon
    roll_lower = roll_mean - rolling_mult * roll_std
    roll_upper = roll_mean + rolling_mult * roll_std
    df_flags.loc[i, "Rolling_lower"] = roll_lower
    df_flags.loc[i, "Rolling_upper"] = roll_upper
    df_flags.loc[i, "Rolling_flag"] = "Yes" if y[i] < roll_lower or y[i] > roll_upper else ""

# --- Display table ---
pd.set_option("display.max_rows", None)
df_flags

In [ ]:
# Convert both to string (safer for IDs)
agreement_df['case_id'] = agreement_df['case_id'].astype(str)
df['case_id'] = df['case_id'].astype(str)

# Now merge
merged_df = agreement_df.merge(
    df[['case_id', 'case_narrative']],
    on='case_id',
    how='inner'
)

merged_df.head()

In [ ]:
import re
import numpy as np
import pandas as pd

# ---------- 1) Clean narrative ----------
def normalize_narrative(s):
    if pd.isna(s): return ""
    s = str(s).strip()
    # remove outer quotes if present
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {"'", '"'}:
        s = s[1:-1]
    # handle both escaped and real CR/LF
    s = s.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    return s

mdf = merged_df.copy()
mdf["narr"] = mdf["case_narrative"].apply(normalize_narrative)

# ---------- 2) Per-case features ----------
# count all 'Drug Name:' occurrences anywhere in text (case-insensitive)
mdf["Num Drugs"] = mdf["narr"].str.count(r"(?i)Drug\s*Name\s*:")

# extract the ADR line (value after the label up to newline or end)
adr_series = mdf["narr"].str.extract(
    r"(?is)Adverse\s*Reaction(?:\(s\))?\s*:\s*(.+?)(?:\n|$)"
)[0].fillna("").str.strip()

def count_adrs(s):
    if not s or s.upper() in {"NA","UNK","UNKNOWN"}:
        return 0
    parts = [p.strip() for p in re.split(r"[;,]", s) if p.strip()]
    return len(parts)

mdf["Num ADRs"] = adr_series.apply(count_adrs)

# first number after 'Age:' / 'Weight:' (NaN if none)
mdf["Age"] = pd.to_numeric(
    mdf["narr"].str.extract(r"(?i)Age\s*:\s*([0-9]+(?:\.[0-9]+)?)")[0],
    errors="coerce"
)
mdf["Weight"] = pd.to_numeric(
    mdf["narr"].str.extract(r"(?i)Weight\s*:\s*([0-9]+(?:\.[0-9]+)?)")[0],
    errors="coerce"
)

# keep one row per case_id (in case merged_df has multiples)
case_base = mdf[["case_id","Num Drugs","Num ADRs","Age","Weight"]].drop_duplicates("case_id")

# ---------- 3) Map cases to samples ----------
if 'samples_meta' in globals() and {'case_id','sample_id'}.issubset(samples_meta.columns):
    case_to_sample = samples_meta[['case_id','sample_id']].drop_duplicates()
else:
    case_to_sample = sampled_df[['case_id','sample_id']].drop_duplicates()

df_join = case_base.merge(case_to_sample, on="case_id", how="inner")

# ---------- 4) Per-sample summary ----------
summary_table = (
    df_join.groupby("sample_id", as_index=False)
           .agg(
               n_cases            = ("case_id", "nunique"),
               mean_num_drugs     = ("Num Drugs", "mean"),
               median_num_drugs   = ("Num Drugs", "median"),
               mean_num_adrs      = ("Num ADRs", "mean"),
               median_num_adrs    = ("Num ADRs", "median"),
               mean_age           = ("Age", "mean"),
               median_age         = ("Age", "median"),
               pct_age_missing    = ("Age", lambda s: 100.0 * s.isna().mean()),
               mean_weight        = ("Weight", "mean"),
               median_weight      = ("Weight", "median"),
               pct_weight_missing = ("Weight", lambda s: 100.0 * s.isna().mean()),
           )
           .sort_values("sample_id")
)

# round for readability
for col in ['mean_num_drugs','median_num_drugs','mean_num_adrs','median_num_adrs',
            'mean_age','median_age','pct_age_missing','mean_weight','median_weight','pct_weight_missing']:
    summary_table[col] = summary_table[col].round(2)

summary_table